# DR Proposals — Colab Runner

Runs Proposals 1–6 (full) and 7–9 (scaffolds) from `experiments/`.

**Honest protocol (enforced in `common.py`):** strict per-fold masking, val-based model selection, balanced (1:1) + imbalanced (1:5) metrics, no GIP, drug content from SMILES (not the degenerate mol2vec).

Each proposal prints per-fold and MEAN `AUROC / AUPR / AUPR(1:5)`.

## 1. Setup: data + dependencies
Clones DRMGNE (base data + SMILES). MeSH/ESM come from the `GCGB/data` folder — upload it next to `DRMGNE/` for the richest features; otherwise the code falls back to similarity-row features automatically.

In [ ]:
import os, sys

# 1) get base data (drug-disease assoc, similarities, SMILES) if missing
if not os.path.isdir('DRMGNE'):
    !git clone --depth 1 https://github.com/Ethereal1z/DRMGNE.git

# 2) dependencies (torch + sklearn are preinstalled on Colab; add rdkit for SMILES->ECFP and 2D/3D)
!pip -q install rdkit scikit-learn requests

# 3) make experiments/ importable. Adjust if your repo lives elsewhere.
EXP_DIR = 'experiments' if os.path.isdir('experiments') else '.'
sys.path.insert(0, os.path.abspath(EXP_DIR))
print('cwd =', os.getcwd())
print('DRMGNE present:', os.path.isdir('DRMGNE'), '| GCGB present:', os.path.isdir('GCGB'))

In [ ]:
import torch
from common import Config, load_data
cfg = Config(epochs=200, gcn_layers=2, emb_dim=256)
print('device =', cfg.device)
# sanity check: load data once (verifies paths + prints feature dims + the mol2vec guard)
_ = load_data(cfg)

## 2. Proposals 1–6 (full, runnable today)
Run in the recommended ROI order. Each `run(cfg)` returns a dict of mean metrics; tweak `cfg.epochs` to trade speed for accuracy.

In [ ]:
import exp6_noisedr;  r6 = exp6_noisedr.run(Config(epochs=200))   # XSimGCL noise-contrastive (lowest effort)

In [ ]:
import exp4_causaldr; r4 = exp4_causaldr.run(Config(epochs=200))  # counterfactual popularity debiasing

In [ ]:
import exp3_hypodr;   r3 = exp3_hypodr.run(Config(epochs=200), K=4)  # disentangled hyperbolic MoA

In [ ]:
import exp5_momoe_dr; r5 = exp5_momoe_dr.run(Config(epochs=200), n_experts=8, top_k=2)  # sparse MoE

In [ ]:
import exp1_diffudr;  r1 = exp1_diffudr.run(Config(epochs=300))   # conditional diffusion (no neg sampling)

In [ ]:
import exp2_semdr;    r2 = exp2_semdr.run(Config())               # TIGER-style generative retrieval

## 3. Part II scaffolds (7–9)
`exp7` runs with RDKit (1D+2D drug modalities). `exp8`/`exp9` need a one-time fetch of STRING/KEGG/AlphaFold (see each file's `run()` printout), then re-run.

In [ ]:
import exp7_m3dr_scaffold; r7 = exp7_m3dr_scaffold.run(Config(epochs=200), use_3d=False)  # 1D+2D drug modalities

In [ ]:
import exp8_biokg_dr_scaffold; r8 = exp8_biokg_dr_scaffold.run(Config(epochs=200))  # prints STRING/KEGG fetch steps

In [ ]:
import exp9_geobind_dr_scaffold; r9 = exp9_geobind_dr_scaffold.run(Config(epochs=200))  # prints AlphaFold fetch steps

## 4. Collect results for the report

In [ ]:
import pandas as pd
res = {'1 DiffuDR': r1, '2 SemDR': r2, '3 HypoDR': r3, '4 CausalDR': r4,
       '5 MoMoE-DR': r5, '6 NoiseDR': r6, '7 M3-DR': r7}
df = pd.DataFrame(res).T[['auroc','auroc_std','aupr','aupr_std','aupr_1to5']]
df.columns = ['AUROC','AUROC_std','AUPR','AUPR_std','AUPR_1to5']
display(df.round(4))
df.round(4).to_csv('experiments/results_summary.csv')